# Notebook 06 — Mini Capstone: Smart FAQ Finder (sample solution)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This is the **sample solution** for the mini capstone in Lesson 11. Try building yours first
from the lesson's checklist; come back here to check your work — no shame in peeking.

**The task:** a pizza shop has a list of FAQ questions and answers. Build a tool where a
customer types a question *in their own words* and gets the right answer — even when the
wording is completely different from the stored question.

## Step 0 — Install the libraries

Run this once. It installs `sentence-transformers` (the model that turns text into vectors) and `chromadb` (the store that holds those vectors and finds the closest match). When it finishes you'll see `Ready.` printed below. On Colab this takes a minute or two the first time.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## Step 1 — The FAQ (ten question → answer pairs)

In [2]:
# Each tuple is one FAQ pair: the customer-style question first, the answer second.
faq = [
    ("What are your opening hours?",
     "We're open 11am to 11pm every day, including weekends."),
    ("Do you offer vegan options?",
     "Yes — we have plant-based cheese and several vegetable-only pizzas."),
    ("Do you deliver?",
     "We deliver free within 5 km; a small fee applies beyond that."),
    ("How much is a large pizza?",
     "A large pizza starts at $14, before any extra toppings."),
    ("Can I book a table for a group?",
     "Yes, call us to reserve a table for groups of six or more."),
    ("Do you have gluten-free crust?",
     "We offer a gluten-free crust on any pizza for a small extra charge."),
    ("What payment methods do you accept?",
     "We take cash, all major cards, and the usual mobile wallets."),
    ("Is there parking nearby?",
     "There's a free public car park right behind the restaurant."),
    ("Do you cater for parties?",
     "Yes, we offer party platters and bulk orders with a day's notice."),
    ("How spicy is the hot pizza?",
     "Our spicy pizza is medium-hot; we can make it milder on request."),
]
# Split the pairs into two parallel lists: questions go into the store as documents,
questions = [q for q, a in faq]
# and answers are kept aside to attach as metadata (the sticky notes).
answers   = [a for q, a in faq]
# Confirm how many pairs we loaded.
print(f"{len(faq)} FAQ pairs loaded.")

10 FAQ pairs loaded.


## Step 2 — Embed the FAQ questions into a store

We store the **questions**, because that's what a customer's typed query should match. Each answer rides along as metadata, like a sticky note attached to its question, so we can hand it back later. When this cell runs it downloads the model the first time, then prints `Indexed 10 FAQ questions.`

In [3]:
import chromadb
from chromadb.utils import embedding_functions

# The embedding function turns each piece of text into a vector of numbers (its meaning).
# all-MiniLM-L6-v2 is a small, free model that runs locally.
minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
# PersistentClient saves the store to a folder on disk so it survives between runs.
client = chromadb.PersistentClient(path="./faq_store")
# Delete any old "faq" collection so re-running this cell starts clean.
try:
    client.delete_collection("faq")
except Exception:
    pass
# Create the collection. We hand it the embedding function so it knows how to turn text
# into vectors, and we ask for cosine distance, which compares vectors by direction
# (meaning) rather than length.
faq_collection = client.create_collection(
    name="faq",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},
)
# Add the FAQ to the store:
faq_collection.add(
    # a unique id for each entry,
    ids=[f"faq_{i}" for i in range(len(faq))],
    # the QUESTIONS are the documents, because a customer types a question and we match
    # their wording against these,
    documents=questions,
    # and each ANSWER is attached as a metadata sticky note that rides along with its
    # question, ready to be returned.
    metadatas=[{"answer": a} for a in answers],
)
# Show how many questions are now stored.
print(f"Indexed {faq_collection.count()} FAQ questions.")

/Users/riteshmodi/gits/leanpub_courses/courses/vectors-and-embeddings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 23285.36it/s]

Indexed 10 FAQ questions.


## Step 3 — The `ask()` function

`ask()` takes a customer's question, finds the single closest stored question, and prints its sticky-note answer along with a similarity score. The test call uses different words from any stored question ("plant-based" instead of "vegan"), yet it lands on the vegan answer with a similarity around 0.51.

In [4]:
def ask(question):
    # Search the store for the single closest stored question (n_results=1 = top match only).
    result = faq_collection.query(query_texts=[question], n_results=1)
    # The result is nested by query then by rank. result["documents"][0][0]:
    # first [0] = our one query, second [0] = its top result.
    matched_question = result["documents"][0][0]
    # Reach into the matched entry's sticky note and read the "answer" key.
    # [0] = our query, [0] = top result, then the "answer" we stored as metadata.
    answer = result["metadatas"][0][0]["answer"]
    # Chroma returns a cosine distance (0 = identical). Subtract from 1 to get a
    # similarity where higher means closer in meaning.
    score = 1 - result["distances"][0][0]
    # Print what was asked, what it matched, and the answer we hand back.
    print(f"You asked : {question}")
    print(f"Matched   : {matched_question}  (similarity {score:.3f})")
    print(f"Answer    : {answer}\n")

# Try it: "plant-based" never appears in the stored questions, yet it finds the vegan one.
ask("Are there plant-based choices on the menu?")

You asked : Are there plant-based choices on the menu?
Matched   : Do you offer vegan options?  (similarity 0.506)
Answer    : Yes — we have plant-based cheese and several vegetable-only pizzas.



## Step 4 — Three reworded questions

None of these reuse the words of the stored questions, yet each finds the right answer. Expect "Until what time can I order tonight?" to match opening hours (similarity around 0.49), "Can you bring it to my house?" to match delivery (around 0.35), and "Where do I leave the car?" to match parking (around 0.45).

In [5]:
# Three more questions worded nothing like the stored ones. Each still lands correctly.
ask("Until what time can I order tonight?")     # -> opening hours
ask("Can you bring it to my house?")            # -> delivery
ask("Where do I leave the car?")                # -> parking

You asked : Until what time can I order tonight?
Matched   : What are your opening hours?  (similarity 0.491)
Answer    : We're open 11am to 11pm every day, including weekends.

You asked : Can you bring it to my house?
Matched   : Do you deliver?  (similarity 0.354)
Answer    : We deliver free within 5 km; a small fee applies beyond that.

You asked : Where do I leave the car?
Matched   : Is there parking nearby?  (similarity 0.452)
Answer    : There's a free public car park right behind the restaurant.



## Recap

- A FAQ finder is the same pattern as semantic search: embed the questions, match a new
  question by meaning, return the stored answer.
- Storing the answer as metadata lets the store hand it back directly.
- "plant-based" matches "vegan", "leave the car" matches "parking" — because the model
  compares meaning, not words. That's the whole idea of the course, working for you.

You've now built two things end to end. 

## Practice — Your Turn

Three short exercises to make the FAQ finder your own. Read each one, make a guess, then run the answer cell below it to check. Each answer cell uses the `ask()` function and the `faq_collection` you already built above, so run the whole notebook top to bottom first.

### Exercise 1 — A question in fresh words

Customers rarely use the same words as your stored FAQ. Ask the finder "Can I pay with my phone?". None of those words appear in the stored questions.

Before you run it, guess which stored FAQ it will match. Try it yourself, then run the answer cell below.

In [6]:
# Answer
# Ask a question worded nothing like the stored FAQ.
# "pay with my phone" should land on the payment-methods FAQ (mobile wallets).
ask("Can I pay with my phone?")

You asked : Can I pay with my phone?
Matched   : What payment methods do you accept?  (similarity 0.367)
Answer    : We take cash, all major cards, and the usual mobile wallets.



### Exercise 2 — Add a brand new FAQ pair

The pizza shop just started doing birthday discounts, so it needs a new FAQ. Add one new question and answer to the collection that is already built, give it a fresh unique id, and store the answer as metadata the same way the others are stored. Then ask a reworded version and confirm the new answer comes back.

Predict what `ask("Is there a deal for birthdays?")` will say before you run it. Try it yourself, then run the answer cell below.

In [7]:
# Answer
# The new FAQ pair we want to teach the finder.
new_question = "Do you give a birthday discount?"
new_answer   = "Yes, show ID on your birthday and your pizza is half price."

# Use a fresh id that does not clash with the existing faq_0..faq_9 ids.
new_id = "faq_birthday"

# Remove this id first if the cell has already been run, so re-running stays clean.
faq_collection.delete(ids=[new_id])

# Add the new pair: the question is the document, the answer rides along as metadata.
faq_collection.add(
    ids=[new_id],                          # the fresh unique id
    documents=[new_question],              # the question is what a customer's words match against
    metadatas=[{"answer": new_answer}],    # the answer is the sticky note we hand back
)

# Show the store grew by one entry.
print(f"Now storing {faq_collection.count()} FAQ questions.\n")

# Ask a reworded version; it should match our new question and return the new answer.
ask("Is there a deal for birthdays?")

Now storing 11 FAQ questions.

You asked : Is there a deal for birthdays?
Matched   : Do you give a birthday discount?  (similarity 0.800)
Answer    : Yes, show ID on your birthday and your pizza is half price.



### Exercise 3 — When the FAQ has no good answer

The finder always returns its closest match, even when nothing in the FAQ truly fits. Ask about something the pizza shop never covers, like whether they sell ice cream. The match will be weak, and a low similarity score is your signal that there is no real answer.

Guess whether the similarity will be high (close to 1) or low. Try it yourself, then run the answer cell below.

In [8]:
# Answer
# Ask something the FAQ does not cover at all.
off_topic = "Do you sell ice cream for dessert?"

# Query the store directly so we can read the raw distance, then turn it into a similarity.
result = faq_collection.query(query_texts=[off_topic], n_results=1)

# Pull out the question it matched and the answer attached to it.
matched_question = result["documents"][0][0]
matched_answer   = result["metadatas"][0][0]["answer"]

# Cosine distance (0 = identical); similarity = 1 - distance, so higher means closer in meaning.
similarity = 1 - result["distances"][0][0]

# Print the closest match and its score.
print(f"You asked : {off_topic}")
print(f"Matched   : {matched_question}  (similarity {similarity:.3f})")
print(f"Answer    : {matched_answer}")

# A low similarity here tells us the FAQ has nothing that really fits this question.
print("\nThe low similarity is the clue: this question has no good answer in the FAQ.")

You asked : Do you sell ice cream for dessert?
Matched   : Do you have gluten-free crust?  (similarity 0.403)
Answer    : We offer a gluten-free crust on any pizza for a small extra charge.

The low similarity is the clue: this question has no good answer in the FAQ.
